In [5]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely import wkb

In [3]:
import seaborn as sns

# Research Area
- City of London
- Westminster
- Hackney
- Camden

# Time Range
- First period: 2016-06-30, 2016-12-31, **2017-06-30**, 2017-12-31, 2018-06-30, 2019-06-30
- Second period: 2022-06-30, 2022-12-31, **2023-06-30**, 2023-12-31, 2024-06-30, 2025-06-30

# First Period Analysis
## Import the data

In [4]:
citylondon_17 = pd.read_csv('data_source/period-2010-to-2017-E09000001-city of london.csv')
camden_17 = pd.read_csv('data_source/period-2010-to-2017-E09000007-camden.csv')
hackney_17 = pd.read_csv('data_source/period-2010-to-2017-E09000012-hackney.csv')
westminster_17 = pd.read_csv('data_source/period-2010-to-2017-E09000033-westminster.csv')

/tmp/ipykernel_39155/722522668.py:1: DtypeWarning: Columns (5,10,11,19,21) have mixed types. Specify dtype option on import or set low_memory=False.
  citylondon_17 = pd.read_csv('data_source/period-2010-to-2017-E09000001-city of london.csv')
/tmp/ipykernel_39155/722522668.py:3: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  hackney_17 = pd.read_csv('data_source/period-2010-to-2017-E09000012-hackney.csv')
/tmp/ipykernel_39155/722522668.py:4: DtypeWarning: Columns (5,10,11,19,21) have mixed types. Specify dtype option on import or set low_memory=False.
  westminster_17 = pd.read_csv('data_source/period-2010-to-2017-E09000033-westminster.csv')


In [6]:
citylondon_17.columns

Index(['filter_period', 'billing_authority_name', 'geocode', 'uarn',
       'billing_reference', 'account_name', 'account_start_date',
       'searchable_address', 'postcode_id', 'geometry', 'occupation_state',
       'occupation_date', 'category_id', 'primary_description',
       'category_subgroup', 'category_group', 'rates_payable',
       'rateable_value', 'total_floor_area', 'unit_of_measure', 'from_date',
       'record_date', 'series', 'epoch'],
      dtype='object')

In [8]:
# Cleaning
citylondon_17N = citylondon_17[citylondon_17['rateable_value'] > 0]  # Remove outliers
citylondon_17N = citylondon_17N.dropna(subset=['rateable_value'])

camden_17N = camden_17[camden_17['rateable_value'] > 0]
camden_17N = camden_17N.dropna(subset=['rateable_value'])

hackney_17N = hackney_17[hackney_17['rateable_value'] > 0]
hackney_17N = hackney_17N.dropna(subset=['rateable_value'])

westminster_17N = westminster_17[westminster_17['rateable_value'] > 0]
westminster_17N = westminster_17N.dropna(subset=['rateable_value'])

# Convert WKB geometry to Shapely objects
citylondon_17N['geom'] = citylondon_17N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
camden_17N['geom'] = camden_17N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
hackney_17N['geom'] = hackney_17N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
westminster_17N['geom'] = westminster_17N['geometry'].apply(lambda x: wkb.loads(x, hex=True))

In [32]:
def extract_all_timepoints_unified(df, revaluation_year='2017'):
    """Extract timepoints for either 2017 or 2023 revaluation"""
    
    if revaluation_year == '2017':
        timepoints = {
            'pre_12m': df[(df['series'] == 2010) & (df['epoch'] == 21)],
            'pre_6m': df[(df['series'] == 2010) & (df['epoch'] == 23)],
            'revaluation': df[(df['series'] == 2017) & (df['epoch'] == 2)],
            'post_6m': df[(df['series'] == 2017) & (df['epoch'] == 4)],
            'post_12m': df[(df['series'] == 2017) & (df['epoch'] == 7)],
            'post_24m': df[(df['series'] == 2017) & (df['epoch'] == 13)]
        }
    elif revaluation_year == '2023':
        timepoints = {
            'pre_12m': df[(df['series'] == 2017) & (df['epoch'] == 32)],
            'pre_6m': df[(df['series'] == 2017) & (df['epoch'] == 34)],
            'revaluation': df[(df['series'] == 2023) & (df['epoch'] == 2)],
            'post_6m': df[(df['series'] == 2023) & (df['epoch'] == 5)],
            'post_12m': df[(df['series'] == 2023) & (df['epoch'] == 8)],
            'post_24m': df[(df['series'] == 2023) & (df['epoch'] == 15)]
        }
    else:
        raise ValueError("revaluation_year must be '2017' or '2023'")
    
    return timepoints

## Dynamic Bunching Analysis
- Tax bracket creep是否真实存在？
  
### Function
- 检测门槛聚集效应：量化企业在税率门槛附近的分布扭曲程度
- 追踪时间演变：观察bunching效应在重估前后的动态变化
- 识别bracket creep：证明税率门槛确实影响企业行为

### 分析机制
- 对比分布密度：比较门槛前**1000英镑**区间 vs 门槛后1000英镑区间的企业数量 \
ratio = (number of businesses in £1000 range before threshold) / (number of businesses in £1000 range after threshold)
- 计算扭曲比例：**ratio < 1**表示门槛前企业"异常稀少"，存在bunching \
distortion_pct = (ratio - 1) * 100
- 量化扭曲程度：distortion_pct显示偏离正常分布的百分比



### <font color ='#DE3163'>可以尝试不同窗口大小：
- window=500   更精细的分析
- window=2000  更宽泛的范围  
- window=1500  中等范围

In [11]:
def dynamic_bunching_analysis(timepoints_dict, thresholds=[15000], window=1000):
    results = []
    
    for period, df in timepoints_dict.items():
        if len(df) == 0:
            continue
            
        df_clean = df.dropna(subset=['rateable_value'])
        
        for threshold in thresholds:
            before = len(df_clean[(df_clean['rateable_value'] >= threshold-window) & 
                                 (df_clean['rateable_value'] < threshold)])
            after = len(df_clean[(df_clean['rateable_value'] >= threshold) & 
                                (df_clean['rateable_value'] <= threshold+window)])
            
            ratio = before / max(after, 1)
            distortion = (ratio - 1) * 100
            
            results.append({
                'period': period,
                'threshold': threshold,
                'before_count': before,
                'after_count': after,
                'ratio': ratio,
                'distortion_pct': distortion
            })
    
    return pd.DataFrame(results)

## Survival analysis
- "哪些企业因为bracket creep而被迫退出市场？"➡️识别可能受bracket creep影响的企业群体
- 如果门槛附近企业退出率更高，证明bracket creep造成了额外压力

### 核心功能
- 追踪企业去向：监测同一批企业在不同时间点是否仍然存在 ➡️ 退出是立即发生还是延迟发生？
- 计算退出率`survival_rate`：量化有多少企业在重估后"消失"了
- 对比分析：比较门槛附近企业 `survival_rate_near_threshold` vs 整体企业的生存表现 `survival_rate_total`

### 分析机制
- 基准建立：以重估前12个月的企业为起始群体
- 生存追踪：检查这些企业的UARN在后续时间点是否还存在
- 分组对比：
整体生存率：所有企业的存活比例 \
门槛附近生存率：15000门槛±window(2000)范围内企业的存活比例

### 关键指标

### <font color ='#DE3163'>可以尝试不同窗口大小

In [12]:
def track_business_survival(timepoints_dict):
    """Track business survival across different time points"""
    
    # Baseline: Use 12 months before revaluation
    baseline = timepoints_dict['pre_12m'].dropna(subset=['rateable_value'])
    baseline_uarns = set(baseline['uarn'])
    
    survival_data = []
    
    for period, df in timepoints_dict.items():
        if len(df) == 0:
            continue
            
        df_clean = df.dropna(subset=['rateable_value'])
        current_uarns = set(df_clean['uarn'])
        
        # Calculate survival rate
        survived = len(baseline_uarns & current_uarns)
        survival_rate = survived / len(baseline_uarns) * 100
        
        # Analyze by threshold groups
        for threshold in [15000]:

            near_threshold = baseline[
                (baseline['rateable_value'] >= threshold-2000) & 
                (baseline['rateable_value'] <= threshold+2000)
            ]
            near_threshold_uarns = set(near_threshold['uarn'])
            
            survived_near = len(near_threshold_uarns & current_uarns)
            survival_rate_near = survived_near / len(near_threshold_uarns) * 100 if len(near_threshold_uarns) > 0 else 0
            
            survival_data.append({
                'period': period,
                'threshold': threshold,
                'total_baseline': len(baseline_uarns),
                'near_threshold_baseline': len(near_threshold_uarns),
                'survived_total': survived,
                'survived_near_threshold': survived_near,
                'survival_rate_total': survival_rate,
                'survival_rate_near_threshold': survival_rate_near
            })
    
    return pd.DataFrame(survival_data)

## Analyze business response strategies by periods
- 企业应对bracket creep是一次性行为还是持续调整过程？
  
### 核心功能
- 时间维度分析：对比企业在重估后6个月 vs 12个月 vs 24 months的不同应对策略
- 策略演变追踪：观察企业应对策略是否随时间变化
- 短期vs长期反应：区分即时反应和深思熟虑后的调整

### 分析价值
1. 策略时间模式
- 即时反应 (6个月)：企业的第一反应，通常是被动接受或立即上诉
- 成熟策略 (12个月)：有时间规划的策略，如空间重组、长期上诉等
短期反应 (6个月)：企业的即时应对策略
中期调整 (12个月)：经过观察和规划的策略修正
- 长期适应 (24个月)：成熟稳定的策略选择

2. 适应过程识别
- 策略转换：从acceptance转为appeal，或从appeal转为space_reduction策略转换 ：从接受转为上诉，或从上诉转为 space_reduction
- 策略稳定性：哪些策略在6-12个月间保持一致
- 学习效应：企业是否在观察其他企业后调整策略

3. Bracket Creep影响的时间差异
- 受影响企业：是否在不同时期采用不同策略来应对15k门槛
- 应对有效性：哪些策略在短期有效，哪些需要长期实施
- 压力持续性：bracket creep的压力是否随时间缓解
  
### <font color ='#DE3163'>标准？策略分类机制
- value减少：Successful Appeal：post值 < reval值的90%（成功降低评估价值）
- area减少：Space Reduction：面积减少超过20%（分割租赁策略）
- Acceptance：post值与reval值差异小于baseline的10%（接受重估）
- Unknown：其他情况

In [18]:
def analyze_business_responses(timepoints_dict):
    """Analyze business response strategies"""
    
    baseline = timepoints_dict['pre_12m'].dropna(subset=['rateable_value'])
    revaluation = timepoints_dict['revaluation'].dropna(subset=['rateable_value'])
    post_6m = timepoints_dict['post_6m'].dropna(subset=['rateable_value'])
    post_12m = timepoints_dict['post_12m'].dropna(subset=['rateable_value'])
    #post_24m = timepoints_dict['post_24m'].dropna(subset=['rateable_value'])
    
    # Merge the data
    baseline_indexed = baseline.set_index('uarn')[['rateable_value', 'total_floor_area']]
    revaluation_indexed = revaluation.set_index('uarn')[['rateable_value', 'total_floor_area']]
    post_6m_indexed = post_6m.set_index('uarn')[['rateable_value', 'total_floor_area']]
    post_12m_indexed = post_12m.set_index('uarn')[['rateable_value', 'total_floor_area']]
    #post_24m_indexed = post_12m.set_index('uarn')[['rateable_value', 'total_floor_area']]
    
    # Find businesses that exist across all time points
    common_uarns = set(baseline_indexed.index) & set(revaluation_indexed.index) & set(post_12m_indexed.index)
    
    all_responses = []
    
    # Analyze responses at 6 months
    common_6m = set(baseline_indexed.index) & set(revaluation_indexed.index) & set(post_6m_indexed.index)
    
    for uarn in common_6m:
        base_val = baseline_indexed.loc[uarn, 'rateable_value']
        reval_val = revaluation_indexed.loc[uarn, 'rateable_value'] 
        post_val = post_6m_indexed.loc[uarn, 'rateable_value']
        
        base_area = baseline_indexed.loc[uarn, 'total_floor_area']
        post_area = post_6m_indexed.loc[uarn, 'total_floor_area']
        
        # Classify response strategies
        strategy = 'unknown'
        
        if post_val < reval_val * 0.9:  # Value decrease
            strategy = 'successful_appeal'
        elif post_area < base_area * 0.8:  # Area reduction
            strategy = 'space_reduction'
        elif abs(post_val - reval_val) < base_val * 0.1:  # Acceptance
            strategy = 'acceptance'
        
        # Check if affected by bracket creep
        affected_15k = (base_val < 15000 and reval_val >= 15000)
        
        all_responses.append({
            'uarn': uarn,
            'period': 'post_6m',
            'baseline_value': base_val,
            'revaluation_value': reval_val,
            'post_value': post_val,
            'baseline_area': base_area,
            'post_area': post_area,
            'strategy': strategy,
            'affected_15k_threshold': affected_15k
        })
    
    # Analyze responses at 12 months
    common_12m = set(baseline_indexed.index) & set(revaluation_indexed.index) & set(post_12m_indexed.index)
    
    for uarn in common_12m:
        base_val = baseline_indexed.loc[uarn, 'rateable_value']
        reval_val = revaluation_indexed.loc[uarn, 'rateable_value'] 
        post_val = post_12m_indexed.loc[uarn, 'rateable_value']
        
        base_area = baseline_indexed.loc[uarn, 'total_floor_area']
        post_area = post_12m_indexed.loc[uarn, 'total_floor_area']
        
        # Classify response strategies
        strategy = 'unknown'
        
        if post_val < reval_val * 0.9:  # Value decrease
            strategy = 'successful_appeal'
        elif post_area < base_area * 0.8:  # Area reduction
            strategy = 'space_reduction'
        elif abs(post_val - reval_val) < base_val * 0.1:  # Acceptance
            strategy = 'acceptance'
        
        # Check if affected by bracket creep
        affected_15k = (base_val < 15000 and reval_val >= 15000)
        
        all_responses.append({
            'uarn': uarn,
            'period': 'post_12m',
            'baseline_value': base_val,
            'revaluation_value': reval_val,
            'post_value': post_val,
            'baseline_area': base_area,
            'post_area': post_area,
            'strategy': strategy,
            'affected_15k_threshold': affected_15k
        })
    
    return pd.DataFrame(all_responses)

In [41]:
def comprehensive_analysis_unified(df, revaluation_year='2017'):
    """Run complete analysis for either 2017 or 2023 revaluation"""
    
    # Extract timepoints based on revaluation year
    timepoints = extract_all_timepoints_unified(df, revaluation_year)
    
    # Run all analyses
    bunching_results = dynamic_bunching_analysis(timepoints)
    survival_results = track_business_survival(timepoints)
    response_results = analyze_business_responses(timepoints)
    #risk_results = classify_exit_risk(timepoints)
    
    return {
        'revaluation_year': revaluation_year,
        'bunching': bunching_results,
        'survival': survival_results,
        'responses': response_results,
        #'risk_classification': risk_results,
        'timepoints_data': timepoints
    }


In [21]:
def create_comparison_tables(all_results):
    """ Create cross-borough comparison tables"""
    
    # Bunching effect comparison
    bunching_comparison = pd.concat([
        results['bunching'].assign(Borough=borough) 
        for borough, results in all_results.items()
    ])
    
    # Survival rate comparison
    survival_comparison = pd.concat([
        results['survival'].assign(Borough=borough) 
        for borough, results in all_results.items()
    ])
    
    # Response strategy comparison
    response_comparison = pd.concat([
        results['responses'].assign(Borough=borough) 
        for borough, results in all_results.items()
    ])
    
    return bunching_comparison, survival_comparison, response_comparison


## Results

In [28]:
# Execute analysis for each borough
camden_results_17 = comprehensive_analysis(camden_17N)
citylondon_results_17 = comprehensive_analysis(citylondon_17N)
hackney_results_17 = comprehensive_analysis(hackney_17N)
westminster_results_17 = comprehensive_analysis(westminster_17N)

# Merge results
all_borough_results_17 = {
    'Camden': camden_results_17,
    'City_of_London': citylondon_results_17, 
    'Hackney': hackney_results_17,
    'Westminster': westminster_results_17
}


# Generate comparative analysis
bunching_all, survival_all, response_all = create_comparison_tables(all_borough_results_17)

summary_stats = bunching_all.groupby(['Borough', 'period'])['distortion_pct'].mean().unstack()
survival_summary = survival_all[survival_all['period'] == 'post_24m'].groupby('Borough')['survival_rate_near_threshold'].mean()

print("Bunching Distortion by Borough:")
print(summary_stats.round(1))

print("\n24-Month Survival Rates:")
print(survival_summary.round(1))

Bunching Distortion by Borough:
period          post_12m  post_24m  post_6m  pre_12m  pre_6m  revaluation
Borough                                                                  
Camden             -11.7     -10.8    -10.7    -16.5   -17.5        -14.0
City_of_London      13.2       8.1     13.4    -13.1   -13.7         15.0
Hackney            -30.2     -30.5    -28.3    -12.0   -12.7        -29.4
Westminster        -10.5     -10.6    -10.3     -5.3    -5.5         -9.6

24-Month Survival Rates:
Borough
Camden            96.3
City_of_London    95.5
Hackney           96.8
Westminster       96.3
Name: survival_rate_near_threshold, dtype: float64


In [25]:
# Generate key insights for all four boroughs
def generate_cross_borough_insights(all_borough_results):
   print("="*80)
   print("CROSS-BOROUGH BRACKET CREEP ANALYSIS")
   print("="*80)
   
   for borough_name, results in all_borough_results.items():
       if results is None:
           continue
           
       print(f"\n{'='*20} {borough_name.upper()} {'='*20}")
       
       print(f"\n1. Bunching Effect Evolution:")
       bunching_summary = results['bunching'].pivot(index='period', columns='threshold', values='distortion_pct')
       print(results['bunching'])
       #print(bunching_summary.round(1))
       
       print(f"\n2. Business Survival Status:")
       survival_key = results['survival'][results['survival']['period'].isin(['revaluation', 'post_6m', 'post_12m', 'post_24m'])]
       survival_summary = survival_key.pivot(index='period', columns='threshold', values='survival_rate_near_threshold')
       print(survival_summary.round(1))
       
       print(f"\n3. Response Strategy Distribution:")
       strategy_counts = results['responses'].groupby('strategy').size()
       strategy_pct = (strategy_counts / strategy_counts.sum() * 100).round(1)
       for strategy, pct in strategy_pct.items():
           print(f"   {strategy}: {strategy_counts[strategy]} businesses ({pct}%)")
       
       print(f"\n4. Bracket Creep Impact:")
       affected_15k = results['responses']['affected_15k_threshold'].sum()
       total_businesses = len(results['responses'])
       affected_pct = (affected_15k / total_businesses * 100) if total_businesses > 0 else 0
       print(f"   Businesses pushed over £15k threshold: {affected_15k} ({affected_pct:.1f}%)")
       print(f"   Total analyzed businesses: {total_businesses}")

   # Cross-borough comparison summary
   print(f"\n{'='*25} COMPARATIVE SUMMARY {'='*25}")
   
   # Bunching distortion comparison (latest period)
   print("\nFinal Bunching Distortion (post_24m):")
   for borough_name, results in all_borough_results.items():
       if results is not None:
           latest_bunching = results['bunching'][results['bunching']['period'] == 'post_24m']
           if len(latest_bunching) > 0:
               distortion = latest_bunching['distortion_pct'].iloc[0]
               print(f"   {borough_name}: {distortion:.1f}%")
   
   # Survival rate comparison
   print("\nFinal Survival Rates (post_24m):")
   for borough_name, results in all_borough_results.items():
       if results is not None:
           latest_survival = results['survival'][results['survival']['period'] == 'post_24m']
           if len(latest_survival) > 0:
               survival_rate = latest_survival['survival_rate_near_threshold'].iloc[0]
               print(f"   {borough_name}: {survival_rate:.1f}%")
   
   # Bracket creep impact comparison
   print("\nBracket Creep Impact Comparison:")
   for borough_name, results in all_borough_results.items():
       if results is not None:
           affected = results['responses']['affected_15k_threshold'].sum()
           total = len(results['responses'])
           impact_rate = (affected / total * 100) if total > 0 else 0
           print(f"   {borough_name}: {affected}/{total} businesses ({impact_rate:.1f}%)")

In [29]:
# Execute cross-borough analysis
generate_cross_borough_insights(all_borough_results_17)

CROSS-BOROUGH BRACKET CREEP ANALYSIS

==================== CAMDEN ====================

1. Bunching Effect Evolution:
        period  threshold  before_count  after_count     ratio  distortion_pct
0      pre_12m      15000           319          382  0.835079      -16.492147
1       pre_6m      15000           315          382  0.824607      -17.539267
2  revaluation      15000           288          335  0.859701      -14.029851
3      post_6m      15000           308          345  0.892754      -10.724638
4     post_12m      15000           309          350  0.882857      -11.714286
5     post_24m      15000           315          353  0.892351      -10.764873

2. Business Survival Status:
threshold    15000
period            
post_12m      97.6
post_24m      96.3
post_6m       98.3
revaluation   93.7

3. Response Strategy Distribution:
   acceptance: 32980 businesses (99.4%)
   space_reduction: 9 businesses (0.0%)
   successful_appeal: 69 businesses (0.2%)
   unknown: 108 businesses

## Second Period Analysis

In [27]:
# import the data
citylondon_23 = pd.read_csv('data_source/period-2017-to-2023-E09000001-city of london.csv')
camden_23 = pd.read_csv('data_source/period-2017-to-2023-E09000007-camden.csv')
hackney_23 = pd.read_csv('data_source/period-2017-to-2023-E09000012-hackney.csv')
westminster_23 = pd.read_csv('data_source/period-2017-to-2023-E09000033-westminster.csv')

# Cleaning
citylondon_23N = citylondon_23[citylondon_23['rateable_value'] > 0]  # Remove outliers
citylondon_23N = citylondon_23N.dropna(subset=['rateable_value'])

camden_23N = camden_23[camden_23['rateable_value'] > 0]
camden_23N = camden_23N.dropna(subset=['rateable_value'])

hackney_23N = hackney_23[hackney_23['rateable_value'] > 0]
hackney_23N = hackney_23N.dropna(subset=['rateable_value'])

westminster_23N = westminster_17[westminster_17['rateable_value'] > 0]
westminster_23N = westminster_17N.dropna(subset=['rateable_value'])

# Convert WKB geometry to Shapely objects
citylondon_23N['geom'] = citylondon_23N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
camden_23N['geom'] = camden_23N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
hackney_23N['geom'] = hackney_23N['geometry'].apply(lambda x: wkb.loads(x, hex=True))
westminster_23N['geom'] = westminster_23N['geometry'].apply(lambda x: wkb.loads(x, hex=True))

/tmp/ipykernel_39155/1951324992.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  citylondon_23 = pd.read_csv('data_source/period-2017-to-2023-E09000001-city of london.csv')
/tmp/ipykernel_39155/1951324992.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  camden_23 = pd.read_csv('data_source/period-2017-to-2023-E09000007-camden.csv')
/tmp/ipykernel_39155/1951324992.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  hackney_23 = pd.read_csv('data_source/period-2017-to-2023-E09000012-hackney.csv')
/tmp/ipykernel_39155/1951324992.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  westminster_23 = pd.read_csv('data_source/period-2017-to-2023-E09000033-westminster.csv')


In [ ]:
time_points_23 = {
    '2022-06-30': 'pre_12m',
    '2022-12-31': 'pre_6m', 
    '2023-06-30': 'revaluation',
    '2023-12-31': 'post_6m',
    '2024-06-30': 'post_12m',
    '2025-06-30': 'post_24m'
}
# Extract data from all time points
def extract_all_timepoints_23(df):
    timepoints = {
        'pre_12m': df[(df['series'] == 2017) & (df['epoch'] == 32)],
        'pre_6m': df[(df['series'] == 2017) & (df['epoch'] == 34)],
        'revaluation': df[(df['series'] == 2023) & (df['epoch'] == 2)],
        'post_6m': df[(df['series'] == 2023) & (df['epoch'] == 5)],
        'post_12m': df[(df['series'] == 2023) & (df['epoch'] == 8)],
        'post_24m': df[(df['series'] == 2023) & (df['epoch'] == 15)]
    }
    return timepoints

In [42]:
def analyze_both_revaluations_separate_data(data_2017, data_2023):
    """Analyze 2017 and 2023 revaluations using separate datasets"""
    
    # Analyze 2017 revaluation using 2017 dataset
    results_2017 = comprehensive_analysis_unified(data_2017, '2017')
    
    # Analyze 2023 revaluation using 2023 dataset  
    results_2023 = comprehensive_analysis_unified(data_2023, '2023')
    
    return {
        '2017_revaluation': results_2017,
        '2023_revaluation': results_2023
    }

# Execute analysis for all boroughs with both periods
camden_both_periods = analyze_both_revaluations_separate_data(camden_17N, camden_23N)
citylondon_both_periods = analyze_both_revaluations_separate_data(citylondon_17N, citylondon_23N)
hackney_both_periods = analyze_both_revaluations_separate_data(hackney_17N, hackney_23N)
westminster_both_periods = analyze_both_revaluations_separate_data(westminster_17N, westminster_23N)

# Combine all results
all_borough_both_periods = {
    'Camden': camden_both_periods,
    'City_of_London': citylondon_both_periods,
    'Hackney': hackney_both_periods,
    'Westminster': westminster_both_periods
}

In [43]:
# Generate comprehensive insights for both revaluation periods
def generate_dual_period_insights(all_borough_both_periods):
   print("="*80)
   print("DUAL REVALUATION ANALYSIS: 2017 vs 2023 BRACKET CREEP IMPACT")
   print("="*80)
   
   for borough_name, periods_data in all_borough_both_periods.items():
       print(f"\n{'='*25} {borough_name.upper()} {'='*25}")
       
       for period, results in periods_data.items():
           period_year = period.split('_')[0]
           print(f"\n--- {period_year} Revaluation Results ---")
           
           # 1. Bunching Effect
           print(f"\n1. Bunching Effect Evolution ({period_year}):")
           bunching_summary = results['bunching'].pivot(index='period', columns='threshold', values='distortion_pct')
           print(bunching_summary.round(1))
           
           # 2. Survival Rates
           print(f"\n2. Business Survival Status ({period_year}):")
           survival_key = results['survival'][results['survival']['period'].isin(['revaluation', 'post_6m', 'post_12m', 'post_24m'])]
           if len(survival_key) > 0:
               survival_summary = survival_key.pivot(index='period', columns='threshold', values='survival_rate_near_threshold')
               print(survival_summary.round(1))
           
           # 3. Response Strategies
           print(f"\n3. Response Strategy Distribution ({period_year}):")
           strategy_counts = results['responses'].groupby('strategy').size()
           strategy_pct = (strategy_counts / strategy_counts.sum() * 100).round(1)
           for strategy, pct in strategy_pct.items():
               print(f"   {strategy}: {strategy_counts[strategy]} businesses ({pct}%)")
           
           # 4. Bracket Creep Impact
           print(f"\n4. Bracket Creep Impact ({period_year}):")
           affected_15k = results['responses']['affected_15k_threshold'].sum()
           total_businesses = len(results['responses'])
           affected_pct = (affected_15k / total_businesses * 100) if total_businesses > 0 else 0
           print(f"   Businesses pushed over £15k threshold: {affected_15k} ({affected_pct:.1f}%)")
           print(f"   Total analyzed businesses: {total_businesses}")
       
       # Compare 2017 vs 2023 for this borough
       print(f"\n--- {borough_name} COMPARISON: 2017 vs 2023 ---")
       
       try:
           # Final bunching distortion comparison
           bunching_2017 = periods_data['2017_revaluation']['bunching']
           bunching_2023 = periods_data['2023_revaluation']['bunching']
           
           final_2017 = bunching_2017[bunching_2017['period'] == 'post_24m']['distortion_pct'].iloc[0]
           final_2023 = bunching_2023[bunching_2023['period'] == 'post_24m']['distortion_pct'].iloc[0]
           
           print(f"Final Bunching Distortion: 2017={final_2017:.1f}% vs 2023={final_2023:.1f}%")
           
           # Bracket creep impact comparison
           impact_2017 = periods_data['2017_revaluation']['responses']['affected_15k_threshold'].sum()
           total_2017 = len(periods_data['2017_revaluation']['responses'])
           impact_2023 = periods_data['2023_revaluation']['responses']['affected_15k_threshold'].sum()
           total_2023 = len(periods_data['2023_revaluation']['responses'])
           
           rate_2017 = (impact_2017/total_2017*100) if total_2017 > 0 else 0
           rate_2023 = (impact_2023/total_2023*100) if total_2023 > 0 else 0
           
           print(f"Bracket Creep Impact: 2017={rate_2017:.1f}% vs 2023={rate_2023:.1f}%")
           
       except Exception as e:
           print(f"Comparison data incomplete: {e}")

   # Cross-borough and cross-period summary
   print(f"\n{'='*30} OVERALL SUMMARY {'='*30}")
   
   print("\nFinal Bunching Distortion Comparison:")
   print("Borough          | 2017  | 2023  | Change")
   print("-" * 45)
   
   for borough_name, periods_data in all_borough_both_periods.items():
       try:
           final_2017 = periods_data['2017_revaluation']['bunching'][periods_data['2017_revaluation']['bunching']['period'] == 'post_24m']['distortion_pct'].iloc[0]
           final_2023 = periods_data['2023_revaluation']['bunching'][periods_data['2023_revaluation']['bunching']['period'] == 'post_24m']['distortion_pct'].iloc[0]
           change = final_2023 - final_2017
           print(f"{borough_name:<15} | {final_2017:>5.1f} | {final_2023:>5.1f} | {change:>+6.1f}")
       except:
           print(f"{borough_name:<15} | Data incomplete")

# Execute the comprehensive dual-period analysis
generate_dual_period_insights(all_borough_both_periods)

DUAL REVALUATION ANALYSIS: 2017 vs 2023 BRACKET CREEP IMPACT

========================= CAMDEN =========================

--- 2017 Revaluation Results ---

1. Bunching Effect Evolution (2017):
threshold    15000
period            
post_12m     -11.7
post_24m     -10.8
post_6m      -10.7
pre_12m      -16.5
pre_6m       -17.5
revaluation  -14.0

2. Business Survival Status (2017):
threshold    15000
period            
post_12m      97.6
post_24m      96.3
post_6m       98.3
revaluation   93.7

3. Response Strategy Distribution (2017):
   acceptance: 32980 businesses (99.4%)
   space_reduction: 9 businesses (0.0%)
   successful_appeal: 69 businesses (0.2%)
   unknown: 108 businesses (0.3%)

4. Bracket Creep Impact (2017):
   Businesses pushed over £15k threshold: 2258 (6.8%)
   Total analyzed businesses: 33166

--- 2023 Revaluation Results ---

1. Bunching Effect Evolution (2023):
threshold    15000
period            
post_12m     -19.5
post_24m     -19.6
post_6m      -19.0
pre_12m      -

KeyError: 'period'